# Tag prospect_sectors with clean SectorFocus values

`database_files/07_prospect_sectors.csv` holds free-text descriptive phrases (e.g. "Logistics / ports / trade corridors"), but `scoring.ts` and `corridor-matching.ts` check for exact matches against the fixed `SectorFocus` list (`"Trade"`, `"Ports"`, `"Logistics"`, ...). Since no phrase is ever exactly equal to one of those tags, those checks silently never fire.

This notebook runs a keyword-matching pass over every descriptive phrase and adds clean tag rows alongside the originals (e.g. "Logistics / ports / trade corridors" → adds `Logistics`, `Ports`, `Trade` as new rows for that prospect). It's idempotent — safe to re-run any time `07_prospect_sectors.csv` is regenerated by `sync_database_files.ipynb` — because it skips rows that are already an exact clean tag, and never inserts a `(prospect_id, tag)` pair that already exists.

This intentionally operates on `database_files/` only, not `Master_List.csv` — see prior discussion on why the enrichment belongs at the operational-data layer, not the archival layer. Run this after `sync_database_files.ipynb` and before `push_to_supabase.ipynb`.

**Known gap, not fixed here:** "Mining" appears in 3 real sector phrases but isn't part of the current 15-value `SectorFocus` type — adding it would also require updating `src/types/index.ts`, which is out of scope for this pass. It's reported below for visibility.

In [1]:
import re
import shutil
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
SECTORS_PATH = DATA_DIR / "database_files" / "07_prospect_sectors.csv"

# Matches src/types/index.ts SectorFocus exactly — do not add values here without
# also updating that type, or the tags will have no code consuming them.
SECTOR_FOCUS_VALUES = {
    "Trade", "Logistics", "Ports", "Maritime", "Energy", "Infrastructure",
    "Real Estate", "Data Infrastructure", "Technology", "Manufacturing",
    "Food / Agribusiness", "Healthcare", "Financial Services",
    "Emerging Markets", "Industrial Platforms",
}

# word-boundary regex patterns, tuned against the actual distinct phrases in the data
# (e.g. plain substring "port" would wrongly fire on "transport"/"support" — hence \b)
KEYWORD_PATTERNS = {
    "Trade": [r"\btrade\b"],
    "Logistics": [r"\blogistics\b", r"\bsupply chain\b", r"\bwarehousing\b", r"\btransport\b", r"\bcargo\b", r"\brail\b"],
    "Ports": [r"\bports?\b"],
    "Maritime": [r"\bmaritime\b", r"\bshipping\b", r"\bvessels?\b", r"\bmarine\b"],
    "Energy": [r"\benergy\b", r"\bfuel\b"],
    "Infrastructure": [r"\binfrastructure\b"],
    "Real Estate": [r"\breal estate\b"],
    "Data Infrastructure": [r"\bdigital infrastructure\b", r"\btelecom\b", r"\bdata (center|centre)s?\b", r"\bfiber\b"],
    "Technology": [r"\btechnology\b", r"\bdigital\b", r"\bsoftware\b", r"\bfintech\b"],
    "Manufacturing": [r"\bmanufactur\w*"],
    "Food / Agribusiness": [r"\bagribusiness\b", r"\bagricultur\w*", r"\bfood\b"],
    "Healthcare": [r"\bhealth\w*"],
    "Financial Services": [r"\bfinancial services\b", r"\bbanking\b", r"\bfinance\b", r"\bfintech\b", r"\bpayments?\b"],
    "Emerging Markets": [r"\bemerging market"],
    "Industrial Platforms": [r"\bindustrial\b", r"\bindustry\b"],
}

sectors_df = pd.read_csv(SECTORS_PATH)
print(sectors_df.shape)
sectors_df.head()

(969, 2)


,prospect_id,sector
0,ca-001,Family office services / private wealth / stew...
1,ca-001,Logistics / ports / trade corridors
2,ca-001,Infrastructure / real estate / development
3,ca-001,Financial services / private capital / investment
4,ca-002,Family office services / private wealth / stew...


## Run the keyword-matching pass

In [2]:
def match_tags(phrase):
    phrase_lower = phrase.lower()
    return [
        tag for tag, patterns in KEYWORD_PATTERNS.items()
        if any(re.search(p, phrase_lower) for p in patterns)
    ]


existing_pairs = set(zip(sectors_df["prospect_id"], sectors_df["sector"]))
new_rows = []
unmatched_phrases = set()

for _, row in sectors_df.iterrows():
    phrase = row["sector"]
    if phrase in SECTOR_FOCUS_VALUES:
        continue  # already a clean tag from a previous run — don't re-scan it

    tags = match_tags(phrase)
    if not tags:
        unmatched_phrases.add(phrase)

    for tag in tags:
        pair = (row["prospect_id"], tag)
        if pair not in existing_pairs:
            new_rows.append({"prospect_id": row["prospect_id"], "sector": tag})
            existing_pairs.add(pair)

print(f"{len(new_rows)} new clean-tag rows to add")
print(f"\n{len(unmatched_phrases)} distinct phrases with no keyword match (expected for 'nature' phrases; check for real gaps like Mining):")
for p in sorted(unmatched_phrases):
    print("  ", p)

0 new clean-tag rows to add

3 distinct phrases with no keyword match (expected for 'nature' phrases; check for real gaps like Mining):
   FDI attraction, investment support
   Family office services / private wealth / stewardship
   Private equity, co-investment, FDI attraction


## Back up and write the enriched sectors table

In [3]:
backup_path = SECTORS_PATH.with_name(SECTORS_PATH.stem + ".pre_tagging.backup.csv")
if not backup_path.exists():
    shutil.copy(SECTORS_PATH, backup_path)
    print("Backed up ->", backup_path)
else:
    print("Backup already exists, skipping ->", backup_path)

enriched_df = pd.concat([sectors_df, pd.DataFrame(new_rows)], ignore_index=True)
enriched_df.to_csv(SECTORS_PATH, index=False)
print("Wrote", SECTORS_PATH.resolve(), "rows:", len(enriched_df), "(was", len(sectors_df), ")")

Backup already exists, skipping -> data\database_files\07_prospect_sectors.pre_tagging.backup.csv
Wrote C:\Users\USER\Desktop\TBP\tbp-dashboard\data\database_files\07_prospect_sectors.csv rows: 969 (was 969 )


## Verify: reload from disk, check tag distribution

In [4]:
reloaded = pd.read_csv(SECTORS_PATH)
print(reloaded.shape)

clean_tags = reloaded[reloaded["sector"].isin(SECTOR_FOCUS_VALUES)]
print(f"\nclean-tag rows: {len(clean_tags)}")
print(clean_tags["sector"].value_counts())

print(f"\nprospects with at least one clean tag: {clean_tags['prospect_id'].nunique()} / {reloaded['prospect_id'].nunique()}")

(969, 2)

clean-tag rows: 600
sector
Logistics               130
Ports                   114
Trade                   113
Infrastructure           46
Financial Services       46
Real Estate              35
Manufacturing            21
Food / Agribusiness      20
Industrial Platforms     18
Technology               17
Energy                   16
Data Infrastructure      14
Healthcare                5
Maritime                  5
Name: count, dtype: int64

prospects with at least one clean tag: 147 / 149
